# Mooncake ToolAgent Trace — Pruning + Statistics

Prunes requests with `input + output > 16384` tokens from `MooncakeToolAgentTrace.csv`,
saves the result as `MooncakeToolAgentTrace_pruned_16384.csv`, and reports statistics.

Loading/statistics functions are identical to `statistics.ipynb` (same CSV format).

In [1]:
import csv
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path
from IPython.display import display, Markdown

ORIGINAL_CSV = "MooncakeToolAgentTrace.csv"
PRUNED_CSV = "MooncakeToolAgentTrace_pruned_16384.csv"
MAX_TOTAL_TOKENS = 16384

## Functions (from statistics.ipynb)

In [2]:
def load_trace(csv_path: str, max_requests: int = None) -> pd.DataFrame:
    """
    Load Azure-format trace CSV into a DataFrame.

    Returns DataFrame with columns:
        timestamp, input_tokens, output_tokens, total_tokens, arrival_sec
    where arrival_sec is relative seconds from the first request.
    """
    rows = []
    with open(csv_path, 'r') as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            if max_requests is not None and i >= max_requests:
                break
            rows.append({
                'timestamp': datetime.fromisoformat(row['TIMESTAMP']),
                'input_tokens': int(row['ContextTokens']),
                'output_tokens': int(row['GeneratedTokens']),
            })

    df = pd.DataFrame(rows)
    df['total_tokens'] = df['input_tokens'] + df['output_tokens']
    df['arrival_sec'] = (df['timestamp'] - df['timestamp'].iloc[0]).dt.total_seconds()

    print(f"Loaded {len(df):,} requests from {Path(csv_path).name}")
    print(f"Time span: {df['timestamp'].iloc[0]} → {df['timestamp'].iloc[-1]}")
    print(f"Duration: {df['arrival_sec'].iloc[-1]:.1f}s ({df['arrival_sec'].iloc[-1]/60:.1f} min)")
    return df


def filter_window(df: pd.DataFrame, start_min: float = 0, end_min: float = None) -> pd.DataFrame:
    """Filter trace to a relative time window (minutes from first request)."""
    first_ts = df['timestamp'].iloc[0]
    t_start = first_ts + timedelta(minutes=start_min)

    if end_min is not None:
        t_end = first_ts + timedelta(minutes=end_min)
        mask = (df['timestamp'] >= t_start) & (df['timestamp'] < t_end)
        label = f"{start_min}–{end_min} min"
    else:
        mask = df['timestamp'] >= t_start
        label = f"{start_min} min–end"

    filtered = df[mask].copy()
    if len(filtered) > 0:
        filtered['arrival_sec'] = (filtered['timestamp'] - filtered['timestamp'].iloc[0]).dt.total_seconds()

    print(f"Window [{label}]: {len(filtered):,} requests")
    return filtered

In [3]:
def _percentile_row(arr, name):
    """Build a statistics row for a numeric array."""
    return {
        "Metric": name,
        "Count": len(arr),
        "Mean": f"{np.mean(arr):.2f}",
        "Std": f"{np.std(arr):.2f}",
        "Min": int(np.min(arr)),
        "P10": f"{np.percentile(arr, 10):.0f}",
        "P25": f"{np.percentile(arr, 25):.0f}",
        "Median": f"{np.median(arr):.0f}",
        "P75": f"{np.percentile(arr, 75):.0f}",
        "P90": f"{np.percentile(arr, 90):.0f}",
        "P99": f"{np.percentile(arr, 99):.0f}",
        "Max": int(np.max(arr)),
    }


def token_stats(df: pd.DataFrame, title: str = None):
    """Display input/output/total token length statistics."""
    if len(df) == 0:
        print("No data.")
        return

    table = pd.DataFrame([
        _percentile_row(df['input_tokens'].values, 'Input tokens'),
        _percentile_row(df['output_tokens'].values, 'Output tokens'),
        _percentile_row(df['total_tokens'].values, 'Total tokens (in+out)'),
    ])

    display(Markdown(f"### {title}" if title else "### Token Length Statistics"))
    display(table.set_index('Metric'))


def arrival_stats(df: pd.DataFrame, title: str = None):
    """Display request arrival rate and inter-arrival time statistics."""
    if len(df) < 2:
        print("Not enough data.")
        return

    arr = df['arrival_sec'].values
    max_sec = int(arr[-1]) + 1

    per_sec = np.zeros(max_sec, dtype=int)
    for t in arr:
        per_sec[min(int(t), max_sec - 1)] += 1

    max_min = int(arr[-1] / 60) + 1
    per_min = np.zeros(max_min, dtype=int)
    for t in arr:
        per_min[min(int(t / 60), max_min - 1)] += 1

    iat_ms = np.diff(arr) * 1000

    rows = [
        _percentile_row(per_sec, 'Arrival rate (req/s)'),
        _percentile_row(per_min, 'Arrival rate (req/min)'),
    ]

    display(Markdown(f"### {title}" if title else "### Arrival Rate Statistics"))
    display(pd.DataFrame(rows).set_index('Metric'))

    if len(iat_ms) > 0:
        display(Markdown("### Inter-Arrival Time"))
        display(pd.DataFrame([_percentile_row(iat_ms, 'Inter-arrival time (ms)')]).set_index('Metric'))


def summary(df: pd.DataFrame, label: str = "Full trace"):
    """Display a one-row summary of the trace/window."""
    if len(df) < 2:
        print("Not enough data.")
        return

    dur = df['arrival_sec'].iloc[-1]
    s = {
        "Window": label,
        "Requests": f"{len(df):,}",
        "Duration (sec)": f"{dur:.1f}",
        "Duration (min)": f"{dur/60:.1f}",
        "Avg req/s": f"{len(df)/dur:.2f}" if dur > 0 else "N/A",
        "Total input tokens": f"{df['input_tokens'].sum():,}",
        "Total output tokens": f"{df['output_tokens'].sum():,}",
        "Avg input len": f"{df['input_tokens'].mean():.1f}",
        "Avg output len": f"{df['output_tokens'].mean():.1f}",
        "Median input len": f"{df['input_tokens'].median():.0f}",
        "Median output len": f"{df['output_tokens'].median():.0f}",
    }

    display(Markdown(f"### Summary — {label}"))
    display(pd.DataFrame([s]).T.rename(columns={0: 'Value'}))

## Pruning: drop requests with input + output > 16384

In [4]:
orig = load_trace(ORIGINAL_CSV)

keep_mask = orig['total_tokens'] <= MAX_TOTAL_TOKENS
pruned = orig[keep_mask].copy()
pruned['arrival_sec'] = (pruned['timestamp'] - pruned['timestamp'].iloc[0]).dt.total_seconds()
removed = orig[~keep_mask]

# Save in the same CSV format (TIMESTAMP,ContextTokens,GeneratedTokens)
out = pd.DataFrame({
    'TIMESTAMP': pruned['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S.%f'),
    'ContextTokens': pruned['input_tokens'],
    'GeneratedTokens': pruned['output_tokens'],
})
out.to_csv(PRUNED_CSV, index=False)
print(f"Saved {len(out):,} requests to {PRUNED_CSV}")

Loaded 23,608 requests from MooncakeToolAgentTrace.csv
Time span: 2024-01-01 00:00:00.282532 → 2024-01-01 00:58:59.831465
Duration: 3539.5s (59.0 min)
Saved 21,073 requests to MooncakeToolAgentTrace_pruned_16384.csv


In [5]:
n_orig, n_pruned, n_removed = len(orig), len(pruned), len(removed)

display(pd.DataFrame([
    {"Trace": "Original", "Requests": n_orig, "Share of original": "100.00%"},
    {"Trace": f"Pruned (total ≤ {MAX_TOTAL_TOKENS})", "Requests": n_pruned,
     "Share of original": f"{100 * n_pruned / n_orig:.2f}%"},
    {"Trace": "Removed", "Requests": n_removed,
     "Share of original": f"{100 * n_removed / n_orig:.2f}%"},
]))

token_share = pd.DataFrame([
    {"Metric": col,
     "Original total": orig[key].sum(),
     "Removed total": removed[key].sum(),
     "Removed share": f"{100 * removed[key].sum() / orig[key].sum():.2f}%"}
    for col, key in [("Input tokens", "input_tokens"),
                     ("Output tokens", "output_tokens"),
                     ("Total tokens", "total_tokens")]
])
display(token_share)

display(Markdown(
    f"**{n_removed:,} / {n_orig:,} requests removed ({100 * n_removed / n_orig:.2f}%) — "
    f"{n_pruned:,} requests remain.**"
))

,Trace,Requests,Share of original
0,Original,23608,100.00%
1,Pruned (total ≤ 16384),21073,89.26%
2,Removed,2535,10.74%


,Metric,Original total,Removed total,Removed share
0,Input tokens,202940084,81870111,40.34%
1,Output tokens,4299817,1054613,24.53%
2,Total tokens,207239901,82924724,40.01%


**2,535 / 23,608 requests removed (10.74%) — 21,073 requests remain.**

## Statistics — pruned trace

Reloaded through `load_trace()` to confirm the saved CSV works with the standard loading util.

In [6]:
df = load_trace(PRUNED_CSV)
token_stats(df, title=f"Token Stats — pruned (total ≤ {MAX_TOTAL_TOKENS})")
arrival_stats(df, title=f"Arrival Rate — pruned (total ≤ {MAX_TOTAL_TOKENS})")
summary(df, label=f"Pruned full trace (total ≤ {MAX_TOTAL_TOKENS})")

Loaded 21,073 requests from MooncakeToolAgentTrace_pruned_16384.csv
Time span: 2024-01-01 00:00:00.282532 → 2024-01-01 00:58:59.831465
Duration: 3539.5s (59.0 min)


### Token Stats — pruned (total ≤ 16384)

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Input tokens,21073,5745.27,3050.36,891,1561,3162,6177,7007,9016,15177,16280
Output tokens,21073,154.00,219.52,1,3,3,26,301,478,821,2000
Total tokens (in+out),21073,5899.26,3061.27,895,1976,3195,6205,7052,9331,15530,16384


### Arrival Rate — pruned (total ≤ 16384)

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Arrival rate (req/s),3540,5.95,2.67,0,3,4,6,8,9,13,16
Arrival rate (req/min),59,357.17,37.55,263,300,334,364,389,404,412,413


### Inter-Arrival Time

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Inter-arrival time (ms),21072,167.97,174.54,0,17,47,114,229,388,814,2240


### Summary — Pruned full trace (total ≤ 16384)

,Value
Window,Pruned full trace (total ≤ 16384)
Requests,"21,073"
Duration (sec),3539.5
Duration (min),59.0
Avg req/s,5.95
Total input tokens,"121,069,973"
Total output tokens,"3,245,204"
Avg input len,5745.3
Avg output len,154.0
Median input len,6177


In [7]:
# Reference: original (unpruned) statistics for comparison
token_stats(orig, title="Token Stats — original (unpruned)")
summary(orig, label="Original full trace")

### Token Stats — original (unpruned)

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Input tokens,23608,8596.24,11002.76,891,1814,3230,6346,7472,16806,61661,126195
Output tokens,23608,182.13,242.28,1,3,13,30,356,507,898,2000
Total tokens (in+out),23608,8778.38,11070.25,895,2206,3333,6403,7566,17157,61963,126527


### Summary — Original full trace

,Value
Window,Original full trace
Requests,"23,608"
Duration (sec),3539.5
Duration (min),59.0
Avg req/s,6.67
Total input tokens,"202,940,084"
Total output tokens,"4,299,817"
Avg input len,8596.2
Avg output len,182.1
Median input len,6346


## Statistics — first N minutes (pruned trace)

Change `WINDOW_MIN` and re-run the cell to inspect a different prefix window.

In [8]:
# First N minutes of the pruned trace
WINDOW_MIN = 10

w = filter_window(df, 0, WINDOW_MIN)
token_stats(w, title=f"Token Stats (0–{WINDOW_MIN} min, pruned)")
arrival_stats(w, title=f"Arrival Rate (0–{WINDOW_MIN} min, pruned)")
summary(w, label=f"0–{WINDOW_MIN} min (pruned)")

Window [0–10 min]: 3,051 requests


### Token Stats (0–10 min, pruned)

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Input tokens,3051,5779.22,2978.28,891,1798,3174,6177,7042,9023,15156,15979
Output tokens,3051,149.97,217.07,1,3,3,26,298,491,786,2000
Total tokens (in+out),3051,5929.19,2995.87,901,2247,3208,6205,7082,9373,15528,16338


### Arrival Rate (0–10 min, pruned)

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Arrival rate (req/s),600,5.08,2.39,0,2,3,5,7,8,11,16
Arrival rate (req/min),10,305.10,26.95,263,272,284,306,324,334,350,352


### Inter-Arrival Time

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Inter-arrival time (ms),3050,196.63,202.61,0,19,55,134,270,453,901,1918


### Summary — 0–10 min (pruned)

,Value
Window,0–10 min (pruned)
Requests,"3,051"
Duration (sec),599.7
Duration (min),10.0
Avg req/s,5.09
Total input tokens,"17,632,392"
Total output tokens,"457,554"
Avg input len,5779.2
Avg output len,150.0
Median input len,6177


In [9]:
# Compare several prefix windows side by side
def compare_prefix_windows(df, minutes_list):
    rows = []
    for end_min in minutes_list:
        w = filter_window(df, 0, end_min)
        if len(w) < 2:
            continue
        dur = w['arrival_sec'].iloc[-1]
        rows.append({
            'Window': f'0–{end_min} min' if end_min is not None else 'full trace',
            'Requests': len(w),
            'Avg req/s': f"{len(w)/dur:.2f}" if dur > 0 else 'N/A',
            'Avg input': f"{w['input_tokens'].mean():.0f}",
            'Med input': f"{w['input_tokens'].median():.0f}",
            'P90 input': f"{np.percentile(w['input_tokens'], 90):.0f}",
            'Max input': int(w['input_tokens'].max()),
            'Avg output': f"{w['output_tokens'].mean():.0f}",
            'Med output': f"{w['output_tokens'].median():.0f}",
            'P90 output': f"{np.percentile(w['output_tokens'], 90):.0f}",
            'Max output': int(w['output_tokens'].max()),
        })
    display(pd.DataFrame(rows).set_index('Window'))


compare_prefix_windows(df, [3, 5, 10, 15, 30, None])

Window [0–3 min]: 907 requests
Window [0–5 min]: 1,561 requests
Window [0–10 min]: 3,051 requests
Window [0–15 min]: 4,773 requests
Window [0–30 min]: 10,050 requests
Window [0 min–end]: 21,073 requests


,Requests,Avg req/s,Avg input,Med input,P90 input,Max input,Avg output,Med output,P90 output,Max output
Window,,,,,,,,,,
0–3 min,907,5.04,5937,6214,9551,15978,152,26,481,1644
0–5 min,1561,5.20,5818,6182,9325,15978,152,26,485,1644
0–10 min,3051,5.09,5779,6177,9023,15979,150,26,491,2000
0–15 min,4773,5.30,5793,6176,9025,16035,148,26,476,2000
0–30 min,10050,5.58,5762,6176,9010,16214,151,26,479,2000
full trace,21073,5.95,5745,6177,9016,16280,154,26,478,2000
